In [1]:
import pandas as pd
import numpy as np
import pathlib

In [2]:
from lsff_utils import data_processing, config_utils

In [3]:
# TODO: Deduplicate this list with the Snakefiles
data_needs = {
    "Country-Vehicle": {
        "Vehicle consumption by WRA -- any": "vehicle_consumption/any",
        'Vehicle "fortifiability" (essentially amount industrially produced)': "vehicle_consumption/fortifiability",
        "Vehicle consumption by WRA -- amount": "vehicle_consumption/amount",
    },
    "Country-Vehicle-Fort": {
        "Vehicle fortification at baseline -- any": "baseline_fortification/any_coverage",
        "Baseline effective % of fortified": "baseline_fortification/effectiveness",
        "Vehicle fortification at baseline -- amount among fortified": "baseline_fortification/concentration",
    },
    "Scenario Definition": {
        "Intervention coverage % of fortifiable": "intervention_fortification/any_coverage",
        "Intervention effective % of fortified": "intervention_fortification/effectiveness",
        "Vehicle fortification in intervention -- amount among fortified": "intervention_fortification/concentration",
    },
}

data_point_names = {
    "mean": "mean",
    "standard deviation": "sd",
}

In [4]:
results_dir = "../results"
pathlib.Path(results_dir).mkdir(parents=True, exist_ok=True)

In [5]:
definition_columns_detailed_values = {
    "wealth_quintile": set(data_processing.WEALTH_QUINTILES),
    "sex": {"Female", "Male"},
}

In [6]:
def expand(data_point_rows, definition_columns):
    for definition_column in definition_columns:
        assert set(data_point_rows[definition_column]) <= (
            set(definition_columns_detailed_values[definition_column])
            | {"all (assumed same)", "total", "All (assumed same)", "Total"}
        )
        # Expand assumptions
        data_point_rows = pd.concat(
            [
                data_point_rows[
                    data_point_rows[definition_column].str.lower()
                    != "all (assumed same)"
                ]
            ]
            + [
                data_point_rows[
                    data_point_rows[definition_column].str.lower()
                    == "all (assumed same)"
                ].assign(
                    **{
                        definition_column: (
                            value.title() if definition_column == "sex" else value
                        )
                    }
                )
                for value in sorted(
                    list(
                        {
                            str(x)
                            for x in definition_columns_detailed_values[
                                definition_column
                            ]
                        }
                        | {"total"}
                    )
                )
            ],
            ignore_index=True,
        )
    return data_point_rows

In [7]:
def reformat_data(data_point_rows):
    columns_to_keep = {
        "Vehicle": "vehicle_name",
        "Sex": "sex",
        "Quintile": "wealth_quintile",
        "Value": "value",
    }
    data_point_rows = data_point_rows[
        [c for c in data_point_rows.columns if c in columns_to_keep.keys()]
    ].rename(
        columns={
            c: new_c
            for c, new_c in columns_to_keep.items()
            if c in data_point_rows.columns
        }
    )
    if "vehicle_name" in data_point_rows.columns:
        data_point_rows["vehicle_name"] = data_point_rows["vehicle_name"].str.lower()
    if "wealth_quintile" in data_point_rows.columns:
        data_point_rows["wealth_quintile"] = (
            data_processing.recode_extraction_wealth_quintile(
                data_point_rows["wealth_quintile"]
            )
        )

    return data_point_rows

In [8]:
def check_totals(data_point_rows, definition_columns, fix=False):
    total_markers = (
        data_point_rows[definition_columns].apply(lambda s: s.str.lower()) == "total"
    )
    total_rows = (total_markers).any(axis=1)
    for _, total_row in (
        data_point_rows[total_rows]
        .assign(num_total_markers=total_markers[total_rows].sum(axis=1))
        .sort_values("num_total_markers", ascending=False)
        .iterrows()
    ):
        matching_rows = pd.Series(True, index=data_point_rows[~total_rows].index)
        for def_column in definition_columns:
            if str(total_row[def_column]).lower() != "total":
                matching_rows = matching_rows & (
                    data_point_rows[~total_rows][def_column] == total_row[def_column]
                )

        if fix:
            data_point_rows.loc[
                (~total_rows) & matching_rows.reindex_like(total_rows), "value"
            ] *= (
                total_row["value"]
                / data_point_rows[~total_rows][matching_rows].value.mean()
            )

        assert np.isclose(
            data_point_rows[~total_rows][matching_rows].value.mean(),
            total_row["value"],
            atol=0,
            rtol=0.1,
        )

    if fix:
        return data_point_rows

In [9]:
def save_results(
    country, sheet_name, sheet_data_needs, fortificant=None, vehicle=None, scenario=None
):
    sheet = pd.read_excel(
        "./Data Extraction Sheet.xlsx", sheet_name=f"{sheet_name} Extraction"
    )
    sheet = sheet[sheet.Country.str.lower() == country]

    if fortificant is not None:
        sheet = sheet[sheet.Fortificant.str.lower() == fortificant]

    if scenario is not None:
        # Strip out special characters and spaces
        sheet = sheet[
            sheet.Scenario.str.lower().str.replace("[\W_]+", "_", regex=True)
            == scenario
        ]

    if vehicle is not None:
        sheet = sheet[sheet.Vehicle.str.lower() == vehicle]

    assert len(sheet) > 0

    for need, short_need_name in sheet_data_needs.items():
        if need not in sheet["Data need"].values:
            # Needs filled by microdata
            print(f"Need {need} not extracted")
            continue

        print(f"Data need: {need}")
        need_rows = sheet[sheet["Data need"] == need]

        if country == "nigeria" and short_need_name == "vehicle_consumption/amount":
            # We do some interpolation here, see below
            continue

        if (
            country == "india"
            and short_need_name == "vehicle_consumption/fortifiability"
        ):
            # This is a special case with more processing to harmonize different source of information;
            # see below
            continue

        if (
            need.endswith("-- any")
            or short_need_name.endswith("_coverage")
            or short_need_name.endswith("effectiveness")
            or short_need_name == "vehicle_consumption/fortifiability"
        ):
            # Percentage
            assert (need_rows.Units == "%").all()
            assert (need_rows["Data point name"] == "percentage").all()
        elif need.endswith("-- amount") or short_need_name.endswith("amount"):
            # Consumption in g/day
            assert (need_rows.Units == "g/day").all()
            assert (
                need_rows["Data point name"].isin(["mean", "standard deviation"])
            ).all()
        elif need.endswith("-- amount among fortified") or short_need_name.endswith(
            "concentration"
        ):
            # Concentration in mcg/g
            assert (need_rows.Units == "mcg/g").all()
            assert (need_rows["Data point name"] == "concentration").all()
        else:
            raise ValueError()

        data_points = need_rows["Data point name"].unique()
        for data_point in data_points:
            data_point_rows = need_rows[need_rows["Data point name"] == data_point]
            data_point_rows = reformat_data(data_point_rows)

            definition_columns = [
                c for c in ["wealth_quintile", "sex"] if c in data_point_rows.columns
            ]

            data_point_rows = expand(data_point_rows, definition_columns)

            def groupby_apply(df, by, func):
                if len(by) > 0:
                    return df.groupby(by).apply(func)
                else:
                    return pd.Series([func(df)])

            check_totals(data_point_rows, definition_columns)

            # if 'wealth_quintile' in definition_columns:
            #     groupby_apply(data_point_rows, [c for c in definition_columns if c != 'wealth_quintile'], lambda df: check_with_total(df[df]))
            # TODO: Check sex totals

            if len(definition_columns) > 0:
                data_point_rows = data_point_rows[
                    (
                        data_point_rows[definition_columns].apply(
                            lambda s: s.str.lower()
                        )
                        != "total"
                    ).all(axis=1)
                ]

            assert (
                groupby_apply(data_point_rows, definition_columns, lambda df: len(df))
                == 1
            ).all()

            if sheet_name == "Country-Vehicle" and "WRA" in need:
                assert (data_point_rows["sex"] == "Female").all()
                data_point_rows["age_start"] = 15
                data_point_rows["age_end"] = 50
            elif "U5" in need:
                data_point_rows["age_start"] = 0
                data_point_rows["age_end"] = 5

            if len(data_points) == 1:
                dir_name = short_need_name
            else:
                dir_name = f"{short_need_name}/{data_point_names[data_point]}"

            file_path = f"{results_dir}{('/' + fortificant) if fortificant is not None else ''}{('/' + vehicle) if vehicle is not None else ''}{('/' + scenario) if scenario is not None else ''}/{dir_name}/{country}.csv"
            pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
            data_point_rows.to_csv(file_path, index=False)

            if short_need_name == "baseline_fortification/any_coverage":
                # NOTE: For extractions, any == full coverage! We assume people are either fully
                # covered or not. This is probably reasonable in Nigeria -- why would people
                # buy multiple different types of bouillon?
                # We do something more sophisticated with India from microdata -- especially
                # relevant because lots of people have ration cards for a certain amount from one source.
                dir_name = "baseline_fortification/full_coverage"
                file_path = f"{results_dir}{('/' + fortificant) if fortificant is not None else ''}{('/' + vehicle) if vehicle is not None else ''}{('/' + scenario) if scenario is not None else ''}/{dir_name}/{country}.csv"
                pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
                data_point_rows.to_csv(file_path, index=False)

In [10]:
for country, vehicle in config_utils.get_configured_combos(["location", "vehicle"]):
    save_results(
        country, "Country-Vehicle", data_needs["Country-Vehicle"], vehicle=vehicle
    )

for country, vehicle, fortificant in config_utils.get_configured_combos(
    ["location", "vehicle", "fortificant"]
):
    save_results(
        country,
        "Country-Vehicle-Fort",
        data_needs["Country-Vehicle-Fort"],
        fortificant=fortificant,
        vehicle=vehicle,
    )

for (
    country,
    vehicle,
    fortificant,
    intervention_scenario,
) in config_utils.get_configured_combos(
    ["location", "vehicle", "fortificant", "intervention_scenario"]
):
    save_results(
        country,
        "Scenario Definition",
        data_needs["Scenario Definition"],
        fortificant=fortificant,
        vehicle=vehicle,
        scenario=intervention_scenario,
    )

/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Need Vehicle consumption by WRA -- any not extracted
Data need: Vehicle "fortifiability" (essentially amount industrially produced)
Need Vehicle consumption by WRA -- amount not extracted
Data need: Vehicle consumption by WRA -- any
Data need: Vehicle "fortifiability" (essentially amount industrially produced)
Data need: Vehicle consumption by WRA -- amount
Data need: Vehicle consumption by WRA -- any
Data need: Vehicle "fortifiability" (essentially amount industrially produced)


Data need: Vehicle consumption by WRA -- amount
Need Vehicle fortification at baseline -- any not extracted
Data need: Baseline effective % of fortified
Data need: Vehicle fortification at baseline -- amount among fortified
Data need: Vehicle fortification at baseline -- any


/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Data need: Baseline effective % of fortified
Data need: Vehicle fortification at baseline -- amount among fortified
Need Vehicle fortification at baseline -- any not extracted
Data need: Baseline effective % of fortified
Data need: Vehicle fortification at baseline -- amount among fortified
Data need: Vehicle fortification at baseline -- any
Data need: Baseline effective % of fortified
Data need: Vehicle fortification at baseline -- amount among fortified


/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Data need: Vehicle fortification at baseline -- any
Data need: Baseline effective % of fortified
Data need: Vehicle fortification at baseline -- amount among fortified
Data need: Intervention coverage % of fortifiable
Data need: Intervention effective % of fortified
Data need: Vehicle fortification in intervention -- amount among fortified
Data need: Intervention coverage % of fortifiable
Data need: Intervention effective % of fortified
Data need: Vehicle fortification in intervention -- amount among fortified
Data need: Intervention coverage % of fortifiable
Data need: Intervention effective % of fortified


/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Data need: Vehicle fortification in intervention -- amount among fortified
Data need: Intervention coverage % of fortifiable
Data need: Intervention effective % of fortified
Data need: Vehicle fortification in intervention -- amount among fortified
Data need: Intervention coverage % of fortifiable
Data need: Intervention effective % of fortified
Data need: Vehicle fortification in intervention -- amount among fortified
Data need: Intervention coverage % of fortifiable
Data need: Intervention effective % of fortified
Data need: Vehicle fortification in intervention -- amount among fortified


/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## Nigeria consumption interpolation/extrapolation

In [11]:
sheet = pd.read_excel(
    "./Data Extraction Sheet.xlsx", sheet_name="Country-Vehicle Extraction"
)
sheet = sheet[sheet.Country.str.lower() == "nigeria"]

/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [12]:
def interpolate_extrapolate(wra_rows, u5_rows):
    wra_rows = reformat_data(wra_rows)
    u5_rows = check_totals(
        expand(reformat_data(u5_rows), ["wealth_quintile", "sex"]),
        ["wealth_quintile", "sex"],
        fix=True,
    )
    u5_rows = (
        u5_rows[
            (u5_rows.sex.astype(str).str.lower() != "total")
            & (u5_rows.wealth_quintile.astype(str).str.lower() != "total")
        ]
        .assign(age_start=0, age_end=5)
        .set_index(["sex", "age_start", "age_end", "wealth_quintile", "vehicle_name"])
        .value
    )

    male_to_female_ratio = u5_rows[
        u5_rows.index.get_level_values("sex") == "Male"
    ].droplevel("sex") / u5_rows[
        u5_rows.index.get_level_values("sex") == "Female"
    ].droplevel(
        "sex"
    )
    assert np.allclose(male_to_female_ratio, male_to_female_ratio.mean())
    male_to_female_ratio = male_to_female_ratio.mean()
    print(f"Male to female ratio: {male_to_female_ratio}")

    adult_rows = (
        pd.concat(
            [
                wra_rows[wra_rows.wealth_quintile.astype(str).str.lower() != "total"],
                wra_rows[
                    wra_rows.wealth_quintile.astype(str).str.lower() != "total"
                ].assign(sex="Male", value=lambda df: df.value * male_to_female_ratio),
            ],
            ignore_index=True,
        )
        .assign(age_start=15, age_end=125)
        .set_index(["sex", "age_start", "age_end", "wealth_quintile", "vehicle_name"])
        .value
    )

    adolescent_rows = (
        (
            adult_rows.droplevel(["age_start", "age_end"]) * 0.5
            + u5_rows.droplevel(["age_start", "age_end"]) * 0.5
        )
        .reset_index()
        .assign(age_start=5, age_end=15)
        .set_index(["sex", "age_start", "age_end", "wealth_quintile", "vehicle_name"])
        .value
    )

    return pd.concat(
        [
            u5_rows,
            adolescent_rows,
            adult_rows,
        ]
    )

In [13]:
mean_consumption = interpolate_extrapolate(
    sheet[
        (sheet["Data need"] == "Vehicle consumption by WRA -- amount")
        & (sheet["Data point name"] == "mean")
    ],
    sheet[
        (sheet["Data need"] == "Vehicle consumption by U5 children -- amount")
        & (sheet["Data point name"] == "mean")
    ],
)
mean_consumption

Male to female ratio: 1.048780487804878


sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                bouillon        8.200000
                            2                bouillon        7.809524
                            3                bouillon        5.759524
                            4                bouillon        4.783333
                            5                bouillon        4.490476
Male    0          5        1                bouillon        8.600000
                            2                bouillon        8.190476
                            3                bouillon        6.040476
                            4                bouillon        5.016667
                            5                bouillon        4.709524
Female  5          15       1                bouillon        8.300000
                            2                bouillon        7.904762
                            3                bouillon        5.829762
                            4   

In [14]:
file_path = f"{results_dir}/bouillon/vehicle_consumption/amount/mean/nigeria.csv"
pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
mean_consumption.to_csv(file_path)

In [15]:
sd_consumption = interpolate_extrapolate(
    sheet[
        (sheet["Data need"] == "Vehicle consumption by WRA -- amount")
        & (sheet["Data point name"] == "standard deviation")
    ],
    sheet[
        (sheet["Data need"] == "Vehicle consumption by U5 children -- amount")
        & (sheet["Data point name"] == "standard deviation")
    ],
)
sd_consumption

Male to female ratio: 1.033333333333333


sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                bouillon        3.570128
                            2                bouillon        3.060109
                            3                bouillon        2.914390
                            4                bouillon        2.404372
                            5                bouillon        1.675774
Male    0          5        1                bouillon        3.689132
                            2                bouillon        3.162113
                            3                bouillon        3.011536
                            4                bouillon        2.484517
                            5                bouillon        1.731633
Female  5          15       1                bouillon        3.599879
                            2                bouillon        3.085610
                            3                bouillon        2.938676
                            4   

In [16]:
file_path = f"{results_dir}/bouillon/vehicle_consumption/amount/sd/nigeria.csv"
pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
sd_consumption.to_csv(file_path)

# Partial coverage amount for Ethiopia and Nigeria

0, since there is no partial coverage in our baseline scenarios here.

In [17]:
for location, vehicle, fortificant in config_utils.get_configured_combos(
    ["location", "vehicle", "fortificant"]
):
    if location == "india":
        continue
    df = pd.DataFrame(
        {
            "wealth_quintile": data_processing.WEALTH_QUINTILES,
            "vehicle_name": vehicle,
            "value": 0,
        }
    )
    for quantity in data_point_names.values():
        path = f"{results_dir}/{fortificant}/{vehicle}/baseline_fortification/partial_coverage_amount/{quantity}/{location}.csv"
        pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
        print(path)
        df.to_csv(path, index=False)

../results/folate/bouillon/baseline_fortification/partial_coverage_amount/mean/nigeria.csv
../results/folate/bouillon/baseline_fortification/partial_coverage_amount/sd/nigeria.csv
../results/iron/bouillon/baseline_fortification/partial_coverage_amount/mean/nigeria.csv
../results/iron/bouillon/baseline_fortification/partial_coverage_amount/sd/nigeria.csv
../results/folate/salt/baseline_fortification/partial_coverage_amount/mean/ethiopia.csv
../results/folate/salt/baseline_fortification/partial_coverage_amount/sd/ethiopia.csv


## India industry consolidation

In [18]:
# We apply the India industry consolidation (overall fortifiability from the extraction sheet) to the rice *not
# distributed by the government*.
# First we disaggregate the industry consolidation number by wealth according to a proxy from HCES: the proportion purchased.

sheet = pd.read_excel(
    "./Data Extraction Sheet.xlsx", sheet_name="Country-Vehicle Extraction"
)
sheet = sheet[sheet.Country.str.lower() == "india"]
assert (sheet.Vehicle.str.lower() == "rice").all()

overall_fortifiability_row = sheet[
    sheet["Data need"]
    == 'Vehicle "fortifiability" (essentially amount industrially produced)'
]
assert len(overall_fortifiability_row) == 1
assert (overall_fortifiability_row.Quintile == "Total").all()

industry_consolidation = float(overall_fortifiability_row.Value.iloc[0])
industry_consolidation

/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


0.0

In [19]:
disparity_proxy = (
    pd.read_csv("../hces/india_rice_fortifiability_disparities.csv")
    .set_index(["sex", "age_start", "age_end", "wealth_quintile"])
    .value
)
disparity_proxy

sex     age_start  age_end  wealth_quintile
Female  0          5        1                  0.503039
                            2                  0.527613
                            3                  0.500180
                            4                  0.430452
                            5                  0.245303
        5          15       1                  0.615470
                            2                  0.606134
                            3                  0.557332
                            4                  0.477339
                            5                  0.270965
        15         30       1                  0.538390
                            2                  0.566315
                            3                  0.509466
                            4                  0.444688
                            5                  0.263671
        30         50       1                  0.581191
                            2                  0.584176
    

In [20]:
# Equally weighting the quintiles, which isn't quite right but close
industry_consolidation_by_quintile = (
    industry_consolidation * disparity_proxy
) / disparity_proxy.mean()
industry_consolidation_by_quintile

sex     age_start  age_end  wealth_quintile
Female  0          5        1                  0.0
                            2                  0.0
                            3                  0.0
                            4                  0.0
                            5                  0.0
        5          15       1                  0.0
                            2                  0.0
                            3                  0.0
                            4                  0.0
                            5                  0.0
        15         30       1                  0.0
                            2                  0.0
                            3                  0.0
                            4                  0.0
                            5                  0.0
        30         50       1                  0.0
                            2                  0.0
                            3                  0.0
                            4         

In [21]:
government_rice = (
    pd.read_csv("../hces/india_proportion_government_rice.csv")
    .set_index(["sex", "age_start", "age_end", "wealth_quintile"])
    .value
)
government_rice

sex     age_start  age_end  wealth_quintile
Female  0          5        1                  0.503039
                            2                  0.527613
                            3                  0.500180
                            4                  0.430452
                            5                  0.245303
        5          15       1                  0.615470
                            2                  0.606134
                            3                  0.557332
                            4                  0.477339
                            5                  0.270965
        15         30       1                  0.538390
                            2                  0.566315
                            3                  0.509466
                            4                  0.444688
                            5                  0.263671
        30         50       1                  0.581191
                            2                  0.584176
    

In [22]:
# We assume all rice distributed by the government is fortifiable.
# The industry consolidation applies to non-government distributed rice.
fortifiability_by_quintile = (
    government_rice + (1 - government_rice) * industry_consolidation_by_quintile
)
fortifiability_by_quintile

sex     age_start  age_end  wealth_quintile
Female  0          5        1                  0.503039
                            2                  0.527613
                            3                  0.500180
                            4                  0.430452
                            5                  0.245303
        5          15       1                  0.615470
                            2                  0.606134
                            3                  0.557332
                            4                  0.477339
                            5                  0.270965
        15         30       1                  0.538390
                            2                  0.566315
                            3                  0.509466
                            4                  0.444688
                            5                  0.263671
        30         50       1                  0.581191
                            2                  0.584176
    

In [23]:
file_path = f"{results_dir}/rice/vehicle_consumption/fortifiability/india.csv"
pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
fortifiability_by_quintile.reset_index().assign(vehicle_name="rice").to_csv(
    file_path, index=False
)